In [ ]:
# Colab setup -- installs SoftMobility when running on Google Colab.
# Tip: For large simulations, switch to a GPU runtime first:
#   Runtime → Change runtime type → T4 GPU (free)
# Then re-run this cell to install the CUDA-enabled JAX.
try:
    import google.colab  # noqa: F401
    import shutil

    if shutil.which("nvidia-smi"):
        %pip install -q -U softmobility "jax[cuda12]"
    else:
        %pip install -q softmobility
except ImportError:
    pass

# Tutorial 01. Defining a soft body assembly

A *SoftBody* instance in SoftMobility is a collection of rigid spheres whose geometry depends on three groups of variables:

| Group | What it is | Example |
|---|---|---|
| **Degrees of freedom (DOFs)** | Internal variables varying during simulation | bending angle, extension |
| **Design parameters** | Fixed geometric / material properties; targets for optimisation | sphere radii, spring stiffness |
| **Inputs** | External signals provided at run time by the solver | gravity, magnetic field |

This tutorial shows two equivalent ways to define a body:
1. **YAML format** — compact, readable, and the recommended approach
2. **Python callables** — for geometry that requires arbitrary Python code

In [ ]:
import os

import numpy as np

import softmobility as sm
from softmobility.classes import figstyle

figstyle.apply()

## 1. YAML format

A YAML string (or file path) is parsed by `SoftBody` into a fully JAX-compatible assembly.
The YAML file has four sections:

```yaml
dof_names:     # registers variable prefixes for DOFs       (optional)
design_names:  # registers variable prefixes for design     (optional)
defaults:      # numeric default values                     (optional)
spheres:       # list of sphere descriptions                (required)
```

Every expression you write in a sphere field is compiled through SymPy into a JAX function, so the assembly can be differentiated and JIT-compiled without any extra work.

The YAML input can be an external file or a Python string.

### Step 1. Two rigid spheres (minimal example)

The simplest possible body: two unit spheres at fixed positions, no degrees of freedom.
This is the classic symmetric dumbbell used in Tutorial 04 (Jeffery orbits).

In [ ]:
yaml_rigid = """
spheres:
  - radius: 1.0
    position: [-1.5, 0, 0]
  - radius: 1.0
    position: [ 1.5, 0, 0]
"""

body_rigid = sm.SoftBody(yaml_rigid, verbose=False)
print(repr(body_rigid))

### Step 2. Adding a degree of freedom

A **DOF** is an internal variable that the solver integrates in time.
Declare a prefix in `dof_names` and use indexed names (e.g., `x0`, `x1`, …) in sphere expressions.

```yaml
dof_names: [x]   # registers prefix "x" → variables x0, x1, x2, ... are DOFs
```

Default values come from the `defaults` section, keyed by the full variable name (`x0: 0.1`).
Variables absent from `defaults` are initialised to zero.

Here `x0` is the rotation angle (in radians) of each sphere around $\boldsymbol{E}_2$.
The checkered sphere visualisation in Part 3 makes this rotation visible.

In [ ]:
yaml_dof = """
dof_names: [x]         # "x" prefix: x, x0, x1... are possible DOF variables

defaults:
  x0: 0.1              # initial tilt angle (rad)

spheres:
  - radius: 1.0
    position: [-1.5, 0, 0]
    orientation: [0, x0, 0]    # Rodrigues vector: sphere 0 tilts by angle x0 around y

  - radius: 1.0
    position: [ 1.5, 0, 0]
    orientation: [0, -x0, 0]   # sphere 1 tilts opposite (antisymmetric)
"""

body_dof = sm.SoftBody(yaml_dof, verbose=False)
print(repr(body_dof))
print()
print("DOF variables:", body_dof.dof_variables)  # ['x0']
print("DOF defaults: ", body_dof.dof_defaults)  # [0.1]

### Step 3. Design parameters

**Design parameters** are fixed during a simulation but are the natural targets for gradient-based optimization (Tutorial 04, Examples 07 and 08).
Declare prefixes in `design_names`.

> ⚠️ **Alphabetical ordering.**  The parser sorts all variable names alphabetically, regardless of the order in `design_names`.  The safest pattern is to always resolve positions by name:
> ```python
> i_radius = body.design_variables.index('radius')
> ```
> then use `design[i_radius]`, the alphabetical ordering then never matters.  If you prefer direct integer indices, check `body.design_variables` first: `design_names: [radius, length, k]` yields `body.design_variables = ['k', 'length', 'radius']`.

In [ ]:
yaml_design = """
dof_names:    [x]
design_names: [radius, length]

defaults:
  x0:     0.1
  radius: 0.5
  length: 2.0

spheres:
  - radius: radius
    position: [-length / 2, 0, 0]
    orientation: [0, x0, 0]

  - radius: radius
    position: [ length / 2, 0, 0]
    orientation: [0, -x0, 0]
"""

body_design = sm.SoftBody(yaml_design, verbose=False)

# Note the alphabetical order — not the order listed in design_names
print("Design variables:", body_design.design_variables)  # ['length', 'radius']
print("Design defaults: ", body_design.design_defaults)  # [ 2.0,  0.5 ]

### Step 4. Forces and torques

The `force` and `torque` entries define the non-hydrodynamic forces applied on spheres, including elastic restoring forces.
The parser extracts the *stiffness coupling matrix* $\mathbf{C}_K$ as the linear operator that relates the DOF vector $\boldsymbol{Q}$ and the six-component elastic force vector (force+torque) $\boldsymbol{f}_E$:
$$\boldsymbol{f}_E = [\boldsymbol
{F}_E,\,\boldsymbol{T}_E] = \mathbf{C}_K(\mathrm{dofs},\mathrm{design})\cdot\boldsymbol{Q}$$
Expressions of force and torque must be **linear in DOFs**, which is the case for linear springs.

A torsion spring of stiffness $k$ is now added to our dumbbell geometry

```
sphere 0:  torque_y = -k * x0   (restoring)
sphere 1:  torque_y = +k * x0   (equal-and-opposite reaction)
```

In [ ]:
yaml_spring = """
dof_names:    [x]
design_names: [radius, length, k]

defaults:
  x0:     0.1
  radius: 0.5
  length: 2.0
  k:      1.0

spheres:
  - radius: radius
    position: [-length / 2, 0, 0]
    orientation: [0, x0, 0]
    torque: [0, -k * x0, 0]     # restoring torque on sphere 0

  - radius: radius
    position: [ length / 2, 0, 0]
    orientation: [0, -x0, 0]
    torque: [0, k * x0, 0]      # equal-and-opposite reaction on sphere 1
"""

body_spring = sm.SoftBody(yaml_spring, verbose=False)
print(repr(body_spring))

### Step 5. Inputs: fields and scalars

**Inputs** are time-varying external fields.
They may only appear in `force` and `torque` expressions, and must be **linear** in the input variables.
The parser extracts the *input coupling matrix* $\mathbf{C}_H$ as the linear operator that relates the input vector $\boldsymbol{H}$ and the six-component external force vector (force+torque) $\boldsymbol{f}_\mathrm{ext}$:
$$\boldsymbol{f}_\mathrm{ext}= [\boldsymbol
{F}_\mathrm{ext},\,\boldsymbol{T}_\mathrm{ext}] = \mathbf{C}_H(\mathrm{dofs},\mathrm{design})\cdot\boldsymbol{H}$$

This formulation is quite flexible and allows for body forces linear in a field (e.g., gravity) or active internal forces.
The YAML parser accepts two kinds of input:

| Kind | YAML names | How to provide at run time |
|---|---|---|
| **3-D field** | base name with suffixes `0`, `1`, `2` (e.g. `gravity0`, `gravity1`, `gravity2`) | `Field` object in `input_map` |
| **Scalar** | base name without suffix (e.g. `act_torque`) | `Scalar` object in `input_map` |

Register the base name in `input_names`.

The file `parameters.yaml` demonstrates both kinds.
Load it below.

In [ ]:
yaml_path = os.path.join(os.path.dirname(sm.__file__), "tutorials", "parameters.yaml")

body_full = sm.SoftBody(yaml_path, verbose=True)
print()
print("DOF variables:    ", body_full.dof_variables)
print("Design variables: ", body_full.design_variables)
print("Input variables:  ", body_full.input_variables)

**Constants**: any name in `defaults` that is *not* declared in `dof_names`, `design_names`, or `input_names` is evaluated once at parse time and substituted numerically into every expression that uses them.

## 2. Python callables

For geometry that requires arbitrary Python / NumPy logic (look-up tables, conditional branches, external data), build the assembly directly from callable objects.

`Sphere` accepts:

| Argument | Signature | Notes |
|---|---|---|
| `radius` | `(dofs, design) → float` | |
| `position` | `(dofs, design, time) → array(3,)` | |
| `orientation` | `(dofs, design, time) → array(3,)` | |
| `force` | `(dofs, design, inputs) → array(3,)` | linear in `inputs` |
| `torque` | `(dofs, design, inputs) → array(3,)` | linear in `inputs` |

In [ ]:
# Build the same 1-DOF symmetric dumbbell from Step 4 using callables
sa = sm.SphereAssembly()

sa.add_dof(name="x0", default=0.1)
sa.add_design(name="length", default=2.0)
sa.add_design(name="radius", default=0.5)
sa.add_design(name="k", default=1.0)

i_x0 = sa.dof_variables.index("x0")
i_length = sa.design_variables.index("length")
i_radius = sa.design_variables.index("radius")
i_k = sa.design_variables.index("k")

sa.add_sphere(
    sm.Sphere(
        radius=lambda dofs, design: design[i_radius],
        position=lambda dofs, design, time: np.array([-design[i_length] / 2, 0.0, 0.0]),
        orientation=lambda dofs, design, time: np.array([0.0, dofs[i_x0], 0.0]),
        torque=lambda dofs, design, inputs: np.array([0.0, -design[i_k] * dofs[i_x0], 0.0]),
    )
)
sa.add_sphere(
    sm.Sphere(
        radius=lambda dofs, design: design[i_radius],
        position=lambda dofs, design, time: np.array([design[i_length] / 2, 0.0, 0.0]),
        orientation=lambda dofs, design, time: np.array([0.0, -dofs[i_x0], 0.0]),
        torque=lambda dofs, design, inputs: np.array([0.0, design[i_k] * dofs[i_x0], 0.0]),
    )
)

print(repr(sa))

The `repr` output confirms that geometry matches the YAML version.